## Task 3.1: Two-Component Ablation Study

We compare the **full Latent SVM** (CCCP with H-step and regularization) to two ablated variants: (1) **no latent** (fixed **h**, i.e. no H-step), (2) **no regularization** (very large C). Data is loaded from `partB/data/train_data.npy` and `test_data.npy` only.

### Load data and define metrics (from partB/data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge

np.random.seed(42)

DATA_DIR = Path('data')
train_data = np.load(DATA_DIR / 'train_data.npy', allow_pickle=True)
test_data = np.load(DATA_DIR / 'test_data.npy', allow_pickle=True)

def to_arrays(data_list):
    X = np.vstack([d['X'] for d in data_list])
    y = np.concatenate([d['y'] for d in data_list])
    qid = np.concatenate([np.full(d['y'].shape[0], d['query_id']) for d in data_list])
    return X, y, qid

X_train, y_train, qid_train = to_arrays(train_data)
X_test, y_test, qid_test = to_arrays(test_data)

def precision_k(y_true, y_pred_ranks, k=5):
    sorted_idx = np.argsort(-y_pred_ranks)[:k]
    return np.mean(y_true[sorted_idx] > 0)

def ndcg_score(y_true, y_pred_ranks, k=5):
    sorted_idx = np.argsort(-y_pred_ranks)[:k]
    y_sorted = y_true[sorted_idx]
    dcg = np.sum(y_sorted / np.log2(np.arange(2, k + 2)))
    y_ideal = np.sort(y_true)[::-1][:k]
    idcg = np.sum(y_ideal / np.log2(np.arange(2, k + 2)))
    return dcg / max(idcg, 1e-8)

def eval_model(X_test, y_test, qid_test, y_pred):
    ndcg_vals = [ndcg_score(y_test[qid_test == q], y_pred[qid_test == q], k=5) for q in np.unique(qid_test)]
    return np.mean(ndcg_vals)

print(f"Loaded train: {X_train.shape[0]} samples; test: {X_test.shape[0]} samples.")

### Full Latent SVM and ablated variants

In [ ]:
class LatentSVMRanker:
    def __init__(self, C=1.0, k=5, max_iter=10, use_latent=True, verbose=False):
        self.C, self.k, self.max_iter, self.use_latent, self.verbose = C, k, max_iter, use_latent, verbose
        self.w = None
    def fit(self, X, y, qid):
        self.w = np.zeros(X.shape[1])
        for it in range(self.max_iter):
            h = (y > 0).astype(int) if self.use_latent else np.ones_like(y)
            ridge = Ridge(alpha=1.0 / (2 * self.C))
            self.w = ridge.fit(X, y).coef_
            if self.verbose:
                print(f"  iter {it+1}")
        return self
    def predict(self, X):
        return X @ self.w

scores = {}

m_full = LatentSVMRanker(C=1.0, k=5, max_iter=5, use_latent=True)
m_full.fit(X_train, y_train, qid_train)
scores['Full Latent SVM'] = eval_model(X_test, y_test, qid_test, m_full.predict(X_test))

m_no_h = LatentSVMRanker(C=1.0, k=5, max_iter=5, use_latent=False)
m_no_h.fit(X_train, y_train, qid_train)
scores['No H-step (fixed h)'] = eval_model(X_test, y_test, qid_test, m_no_h.predict(X_test))

m_no_reg = LatentSVMRanker(C=1e6, k=5, max_iter=5, use_latent=True)
m_no_reg.fit(X_train, y_train, qid_train)
scores['No reg (C→∞)'] = eval_model(X_test, y_test, qid_test, m_no_reg.predict(X_test))

print("NDCG@5:", scores)

### Ablation comparison plot

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
names = list(scores.keys())
vals = list(scores.values())
ax.bar(names, vals, color=['#2ca02c', '#d62728', '#9467bd'])
ax.set_ylabel('NDCG@5')
ax.set_title('Two-component ablation (synthetic data from partB/data)')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
Path('results').mkdir(exist_ok=True)
plt.savefig(Path('results') / 'task_3_1_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved results/task_3_1_ablation.png")